In [1]:
import pandas as pd
import numpy as np
import zipfile
import re
import os

https://measureofamerica.org/wp-content/uploads/2009/05/measureofamerica_methodology.pdf

calculating HDI-like indices

Smth further useful:
https://www.ers.usda.gov/data-products/county-level-data-sets/county-level-data-sets-download-data

# Income - Economy

### SAIPE - Median Income

In [2]:
zip_path = "saipe_medianincome.zip"
all_dfs = []

with zipfile.ZipFile(zip_path, "r") as z:
    xlsx_files = [
        f for f in z.namelist()
        if f.endswith(".xlsx") and "MACOSX" not in f and ".DS_Store" not in f
    ]

    for file in xlsx_files:

        # extract year from filename: est14all.xlsx → 2014
        match = re.search(r'est(\d{2})all', file)
        if match:
            year = int("20" + match.group(1))
        else:
            match4 = re.search(r'(19|20)\d{2}', file)
            year = int(match4.group(0)) if match4 else None

        with z.open(file) as f:
            df = pd.read_excel(f, header=1,
                               dtype={ "State FIPS Code": str,
                                       "County FIPS Code": str
                                     })

        df["year"] = year
        all_dfs.append(df)

df_all = pd.concat(all_dfs, ignore_index=True)
df_all["year"] = df_all["year"].astype(int)
df_all = df_all.sort_values(["year", "State FIPS Code", "County FIPS Code"]).reset_index(drop=True)

In [3]:
df_all["STCOFIPS"] = df_all["State FIPS Code"].astype(str).str.zfill(2)+df_all["County FIPS Code"].astype(str).str.zfill(3)

df_all = df_all.rename(columns={'Postal Code':'State'})

In [4]:
for col in df_all.columns:
    if col not in ["State FIPS Code", "County FIPS Code", "State", "Name", "year", "STCOFIPS"]:
        df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

In [5]:
df_all

,State FIPS Code,County FIPS Code,State,Name,"Poverty Estimate, All Ages",90% CI Lower Bound,90% CI Upper Bound,"Poverty Percent, All Ages",90% CI Lower Bound.1,90% CI Upper Bound.1,...,90% CI Lower Bound.6,90% CI Upper Bound.6,"Poverty Estimate, Age 0-4",90% CI Lower Bound.7,90% CI Upper Bound.7,"Poverty Percent, Age 0-4",90% CI Lower Bound.8,90% CI Upper Bound.8,year,STCOFIPS
0,00,0,US,United States,39108422.0,38858742.0,39358102.0,13.2,13.1,13.3,...,51956.0,52102.0,4369698.0,4311741.0,4427655.0,21.2,20.9,21.5,2008,00000
1,01,0,AL,Alabama,721875.0,706983.0,736766.0,15.9,15.6,16.2,...,41924.0,43248.0,82291.0,78070.0,86511.0,27.0,25.6,28.4,2008,01000
2,01,1,AL,Autauga County,5366.0,4417.0,6316.0,10.7,8.8,12.6,...,48564.0,54679.0,NaN,NaN,NaN,NaN,NaN,NaN,2008,01001
3,01,101,AL,Montgomery County,37511.0,33283.0,41740.0,17.6,15.6,19.6,...,41116.0,45478.0,NaN,NaN,NaN,NaN,NaN,NaN,2008,01101
4,01,103,AL,Morgan County,14284.0,12322.0,16246.0,12.5,10.8,14.2,...,42902.0,49666.0,NaN,NaN,NaN,NaN,NaN,NaN,2008,01103
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51122,56,037,WY,Sweetwater County,4056.0,3278.0,4834.0,10.0,8.1,11.9,...,69128.0,83756.0,NaN,NaN,NaN,NaN,NaN,NaN,2023,56037
51123,56,039,WY,Teton County,1429.0,1108.0,1750.0,6.2,4.8,7.6,...,113436.0,146876.0,NaN,NaN,NaN,NaN,NaN,NaN,2023,56039
51124,56,041,WY,Uinta County,2109.0,1643.0,2575.0,10.3,8.0,12.6,...,70796.0,87376.0,NaN,NaN,NaN,NaN,NaN,NaN,2023,56041
51125,56,043,WY,Washakie County,929.0,727.0,1131.0,12.3,9.6,15.0,...,55169.0,70419.0,NaN,NaN,NaN,NaN,NaN,NaN,2023,56043


In [6]:
df_all.to_parquet('saipe_medianincome.parquet')

In [7]:
df_all.columns

Index(['State FIPS Code', 'County FIPS Code', 'State', 'Name',
       'Poverty Estimate, All Ages', '90% CI Lower Bound',
       '90% CI Upper Bound', 'Poverty Percent, All Ages',
       '90% CI Lower Bound.1', '90% CI Upper Bound.1',
       'Poverty Estimate, Age 0-17', '90% CI Lower Bound.2',
       '90% CI Upper Bound.2', 'Poverty Percent, Age 0-17',
       '90% CI Lower Bound.3', '90% CI Upper Bound.3',
       'Poverty Estimate, Age 5-17 in Families', '90% CI Lower Bound.4',
       '90% CI Upper Bound.4', 'Poverty Percent, Age 5-17 in Families',
       '90% CI Lower Bound.5', '90% CI Upper Bound.5',
       'Median Household Income', '90% CI Lower Bound.6',
       '90% CI Upper Bound.6', 'Poverty Estimate, Age 0-4',
       '90% CI Lower Bound.7', '90% CI Upper Bound.7',
       'Poverty Percent, Age 0-4', '90% CI Lower Bound.8',
       '90% CI Upper Bound.8', 'year', 'STCOFIPS'],
      dtype='object')

In [8]:
df_all=df_all[['STCOFIPS', 'year', 'Poverty Percent, All Ages', '90% CI Lower Bound.1', '90% CI Upper Bound.1', 
        'Median Household Income', '90% CI Lower Bound.6', '90% CI Upper Bound.6']]

In [9]:
rename_cols = {
    "STCOFIPS": "STCOFIPS",
    "Poverty Percent, All Ages": "poverty_rate_all",
    "90% CI Lower Bound.1": "poverty_rate_all_lb90",
    "90% CI Upper Bound.1": "poverty_rate_all_ub90",
    "Median Household Income": "median_household_income",
    "90% CI Lower Bound.6": "median_hh_income_lb90",
    "90% CI Upper Bound.6": "median_hh_income_ub90",
}

df_all = df_all.rename(columns=rename_cols)


In [10]:
df_all=df_all.melt(
    id_vars=["STCOFIPS", "year"],
    var_name="Description",
    value_name="value"
)


unit_map = {
    "poverty_rate_all": "Percent",
    "poverty_rate_all_lb90": "Percent",
    "poverty_rate_all_ub90": "Percent",
    "median_household_income": "Dollars",
    "median_hh_income_lb90": "Dollars",
    "median_hh_income_ub90": "Dollars",
}

df_all["Unit"] = df_all["Description"].map(unit_map)

In [11]:
df_all = df_all[['STCOFIPS', 'Description', 'Unit', 'year', 'value']]
df_all = df_all[df_all['year']!=2023].reset_index(drop=True)

### BEA - GDP

In [12]:
zip_path = "bea_economic_data/CAGDP1.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()
    print(files)


['CAGDP1__ALL_AREAS_2001_2023.csv', 'CAGDP1__definition.xml', 'CAGDP1__Footnotes.html', 'CAGDP1_AK_2001_2023.csv', 'CAGDP1_AL_2001_2023.csv', 'CAGDP1_AR_2001_2023.csv', 'CAGDP1_AZ_2001_2023.csv', 'CAGDP1_CA_2001_2023.csv', 'CAGDP1_CO_2001_2023.csv', 'CAGDP1_CT_2001_2023.csv', 'CAGDP1_DC_2001_2023.csv', 'CAGDP1_DE_2001_2023.csv', 'CAGDP1_FL_2001_2023.csv', 'CAGDP1_GA_2001_2023.csv', 'CAGDP1_HI_2001_2023.csv', 'CAGDP1_IA_2001_2023.csv', 'CAGDP1_ID_2001_2023.csv', 'CAGDP1_IL_2001_2023.csv', 'CAGDP1_IN_2001_2023.csv', 'CAGDP1_KS_2001_2023.csv', 'CAGDP1_KY_2001_2023.csv', 'CAGDP1_LA_2001_2023.csv', 'CAGDP1_MA_2001_2023.csv', 'CAGDP1_MD_2001_2023.csv', 'CAGDP1_ME_2001_2023.csv', 'CAGDP1_MI_2001_2023.csv', 'CAGDP1_MN_2001_2023.csv', 'CAGDP1_MO_2001_2023.csv', 'CAGDP1_MS_2001_2023.csv', 'CAGDP1_MSA_2001_2023.csv', 'CAGDP1_MT_2001_2023.csv', 'CAGDP1_NC_2001_2023.csv', 'CAGDP1_ND_2001_2023.csv', 'CAGDP1_NE_2001_2023.csv', 'CAGDP1_NH_2001_2023.csv', 'CAGDP1_NJ_2001_2023.csv', 'CAGDP1_NM_2001_2023

In [13]:
with zipfile.ZipFile(zip_path, "r") as z:
    first_file = z.namelist()[0]
    print("Reading:", first_file)

    with z.open(first_file) as f:
        df = pd.read_csv(f, encoding="latin1")


Reading: CAGDP1__ALL_AREAS_2001_2023.csv


In [14]:
df = df.iloc[:-4]
df["GeoFIPS"] = df["GeoFIPS"].str.replace('"', '', regex=False)


In [15]:
df['Description'].unique()

array(['Real GDP (thousands of chained 2017 dollars) ',
       'Chain-type quantity indexes for real GDP ',
       'Current-dollar GDP (thousands of current dollars) '], dtype=object)

In [16]:
df_gdp = (df
          [['GeoFIPS', 'Description', 'Unit', 
            '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', 
            '2016', '2017', '2018', '2019', '2020', '2021', '2022']]
         )

In [17]:
df_gdp[df_gdp['Description'].str.contains('Real GDP')][['GeoFIPS', '2017']]

,GeoFIPS,2017
0,00000,19612102000
3,01000,216615470
6,01001,1762558
9,01003,7382558
12,01005,761761
...,...,...
9519,94000,1250759343
9522,95000,4210089181
9525,96000,2286160711
9528,97000,681310123


### BEA - Personal Income

In [18]:
zip_path = "bea_economic_data/CAINC4.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()
    print(files)


['CAINC4__ALL_AREAS_1969_2023.csv', 'CAINC4__definition.xml', 'CAINC4__Footnotes.html', 'CAINC4_AK_1969_2023.csv', 'CAINC4_AL_1969_2023.csv', 'CAINC4_AR_1969_2023.csv', 'CAINC4_AZ_1969_2023.csv', 'CAINC4_CA_1969_2023.csv', 'CAINC4_CO_1969_2023.csv', 'CAINC4_CSA_1969_2023.csv', 'CAINC4_CT_1969_2023.csv', 'CAINC4_DC_1969_2023.csv', 'CAINC4_DE_1969_2023.csv', 'CAINC4_FL_1969_2023.csv', 'CAINC4_GA_1969_2023.csv', 'CAINC4_HI_1969_2023.csv', 'CAINC4_IA_1969_2023.csv', 'CAINC4_ID_1969_2023.csv', 'CAINC4_IL_1969_2023.csv', 'CAINC4_IN_1969_2023.csv', 'CAINC4_KS_1969_2023.csv', 'CAINC4_KY_1969_2023.csv', 'CAINC4_LA_1969_2023.csv', 'CAINC4_MA_1969_2023.csv', 'CAINC4_MD_1969_2023.csv', 'CAINC4_MDIV_1969_2023.csv', 'CAINC4_ME_1969_2023.csv', 'CAINC4_MI_1969_2023.csv', 'CAINC4_MIC_1969_2023.csv', 'CAINC4_MN_1969_2023.csv', 'CAINC4_MO_1969_2023.csv', 'CAINC4_MS_1969_2023.csv', 'CAINC4_MSA_1969_2023.csv', 'CAINC4_MT_1969_2023.csv', 'CAINC4_NC_1969_2023.csv', 'CAINC4_ND_1969_2023.csv', 'CAINC4_NE_1969_

In [19]:
with zipfile.ZipFile(zip_path, "r") as z:
    first_file = z.namelist()[0]
    print("Reading:", first_file)

    with z.open(first_file) as f:
        df = pd.read_csv(f, encoding="latin1")


Reading: CAINC4__ALL_AREAS_1969_2023.csv


/tmp/ipykernel_10884/3276347168.py:6: DtypeWarning: Columns (2,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, encoding="latin1")


In [20]:
df = df.iloc[:-4]
df["GeoFIPS"] = df["GeoFIPS"].str.replace('"', '', regex=False)


In [21]:
df['Description'].unique()

array(['Personal income (thousands of dollars) ',
       ' Nonfarm personal income 1/', ' Farm income 2/',
       'Population (persons) 3/',
       'Per capita personal income (dollars) 4/',
       'Earnings by place of work ',
       'Less: Contributions for government social insurance 5/',
       ' Employee and self-employed contributions for government social insurance ',
       ' Employer contributions for government social insurance ',
       'Plus: Adjustment for residence 6/',
       'Equals: Net earnings by place of residence ',
       'Plus: Dividends, interest, and rent 7/',
       'Plus: Personal current transfer receipts ', 'Wages and salaries ',
       'Supplements to wages and salaries ',
       ' Employer contributions for employee pension and insurance funds 8/',
       "Proprietors' income 9/", " Farm proprietors' income ",
       " Nonfarm proprietors' income "], dtype=object)

In [22]:
df[df['Description'].str.contains('Plus: Dividends, interest, and rent')]

,GeoFIPS,GeoName,Region,TableName,LineCode,IndustryClassification,Description,Unit,1969,1970,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
11,00000,United States,,CAINC4,46.0,...,"Plus: Dividends, interest, and rent 7/",Thousands of dollars,119140000,133564000,...,2771030000,2941428000,3030442000,3249424000,3455566000,3633001000,3645515000,3948422000,4339000000,4806534000
31,01000,Alabama,5,CAINC4,46.0,...,"Plus: Dividends, interest, and rent 7/",Thousands of dollars,1153840,1365531,...,31182508,32794975,33393090,35473784,36450100,38353121,38819931,44062052,46467249,51025653
51,01001,"Autauga, AL",5,CAINC4,46.0,...,"Plus: Dividends, interest, and rent 7/",Thousands of dollars,7626,9367,...,326404,337496,356294,361008,365377,379112,380836,406990,407792,446704
71,01003,"Baldwin, AL",5,CAINC4,46.0,...,"Plus: Dividends, interest, and rent 7/",Thousands of dollars,23653,28828,...,1704465,1822393,1904665,2125439,2181653,2419259,2467796,2889743,3173002,3462188
91,01005,"Barbour, AL",5,CAINC4,46.0,...,"Plus: Dividends, interest, and rent 7/",Thousands of dollars,6010,6998,...,122618,123847,128760,125057,121916,133237,132787,138608,144676,156067
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63911,94000,Plains,4.0,CAINC4,46.0,...,"Plus: Dividends, interest, and rent 7/",Thousands of dollars,9019630,10113552,...,181223223,188996557,194560176,200733423,214307066,223514295,227756475,256807100,278544325,306622322
63931,95000,Southeast,5.0,CAINC4,46.0,...,"Plus: Dividends, interest, and rent 7/",Thousands of dollars,20074735,23440086,...,660665594,702542783,723683691,785912707,837808072,890611225,902293004,982597903,1077408741,1193918693
63951,96000,Southwest,6.0,CAINC4,46.0,...,"Plus: Dividends, interest, and rent 7/",Thousands of dollars,8603870,10062097,...,322868770,332192119,325950389,359515629,387307729,412148079,411510262,456529555,510469197,563791376
63971,97000,Rocky Mountain,7.0,CAINC4,46.0,...,"Plus: Dividends, interest, and rent 7/",Thousands of dollars,2997754,3472483,...,110253582,118426800,122137017,131258336,140305845,152523013,156439510,182391400,200080765,221592174


In [23]:
df_pi = (df
         [df['Description'].isin(['Personal income (thousands of dollars) ', 'Population (persons) 3/', 'Per capita personal income (dollars) 4/', 'Plus: Dividends, interest, and rent 7/'])]
         [['GeoFIPS', 'Description', 'Unit', 
           '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', 
           '2016', '2017', '2018', '2019', '2020', '2021', '2022']]
        )

### Combining all

In [24]:
df_merged = pd.concat([df_gdp, df_pi], ignore_index=True)
df_merged=df_merged.sort_values(['GeoFIPS']).reset_index(drop=True)

rename_map = {
    "Real GDP (thousands of chained 2017 dollars) ": "real_gdp_chained_2017",
    "Current-dollar GDP (thousands of current dollars) ": "nominal_gdp",
    "Chain-type quantity indexes for real GDP ": "real_gdp_quantity_index",
    "Personal income (thousands of dollars) ": "personal_income",
    "Per capita personal income (dollars) 4/": "personal_income_pc",
    "Population (persons) 3/": "population",
    "Plus: Dividends, interest, and rent 7/": "dividends",
}

df_merged["Description"] = df_merged["Description"].replace(rename_map)

df_merged = df_merged.rename(columns={'GeoFIPS':'STCOFIPS'})

In [25]:
df_merged = df_merged.melt(
    id_vars=["STCOFIPS", "Description", "Unit"],
    var_name="year",
    value_name="value"
)


In [26]:
df_combined=pd.concat([df_merged, df_all], ignore_index=True)
df_combined=df_combined.sort_values(['STCOFIPS', 'year']).reset_index(drop=True)

In [27]:
df_combined["STCOFIPS"] = df_combined["STCOFIPS"].astype(str).str.strip()
df_combined = df_combined[
    df_combined["STCOFIPS"].astype(str).str.fullmatch(r"\d{5}")
].reset_index(drop=True)

In [28]:
df_combined["year"] = df_combined["year"].astype(str).str.strip()
df_combined = df_combined[
    df_combined["year"].str.fullmatch(r"\d{4}")
]
df_combined["year"] = df_combined["year"].astype(int)

df_combined["value"] = pd.to_numeric(df_combined["value"], errors="coerce")

In [29]:
df_combined.to_parquet('income_combined.parquet')

In [30]:
df_combined

,STCOFIPS,Description,Unit,year,value
0,00000,real_gdp_chained_2017,Thousands of chained 2017 dollars,2008,1.678148e+10
1,00000,personal_income_pc,Dollars,2008,4.085400e+04
2,00000,dividends,Thousands of dollars,2008,2.438478e+09
3,00000,population,Number of persons,2008,3.040940e+08
4,00000,personal_income,Thousands of dollars,2008,1.242341e+10
...,...,...,...,...,...
622519,56045,poverty_rate_all_lb90,Percent,2022,8.500000e+00
622520,56045,poverty_rate_all_ub90,Percent,2022,1.450000e+01
622521,56045,median_household_income,Dollars,2022,6.767700e+04
622522,56045,median_hh_income_lb90,Dollars,2022,6.086700e+04


In [31]:
df_income = df_combined[df_combined["Description"] == "median_household_income"].copy()

df_income["value"] = pd.to_numeric(df_income["value"], errors="coerce")

df_income["income_index"] = (
    df_income.groupby("year")["value"]
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()) * 10)
)

# Match lower / upper uncertainty estimates
df_income["lower"] = pd.to_numeric(
    df_combined[df_combined["Description"] == "median_hh_income_lb90"]["value"].values,
    errors="coerce"
)
df_income["upper"] = pd.to_numeric(
    df_combined[df_combined["Description"] == "median_hh_income_ub90"]["value"].values,
    errors="coerce"
)

# Penalty
lam = 0.35

# Adjusted value = midpoint – penalty * CI width
df_income["value_adj"] = (df_income["value"]) - lam * (df_income["upper"] - df_income["lower"])

df_income["income_index_adj"] = (
    df_income.groupby("year")["value_adj"]
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()) * 10)
)

df_income = df_income.reset_index(drop=True)

In [32]:
df_income.groupby('year').size()

year
2008    3194
2009    3195
2010    3195
2011    3195
2012    3195
2013    3195
2014    3194
2015    3194
2016    3194
2017    3194
2018    3194
2019    3194
2020    3195
2021    3195
2022    3196
dtype: int64

In [33]:
df_income.to_parquet('median_income_combined.parquet')

# Education

### Educational Attainment

https://data.census.gov/table/ACSST1Y2023.S1501?q=Educational+Attainment&g=010XX00US$0500000

In [34]:
zip_path = "ACSST1Y2010.S1501.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()
    print(files)

['ACSST1Y2024.S1501-Data.csv', 'ACSST1Y2024.S1501-Column-Metadata.csv', 'ACSST1Y2024.S1501-Table-Notes.txt', 'ACSST1Y2023.S1501-Data.csv', 'ACSST1Y2023.S1501-Column-Metadata.csv', 'ACSST1Y2023.S1501-Table-Notes.txt', 'ACSST1Y2022.S1501-Data.csv', 'ACSST1Y2022.S1501-Column-Metadata.csv', 'ACSST1Y2022.S1501-Table-Notes.txt', 'ACSST1Y2021.S1501-Data.csv', 'ACSST1Y2021.S1501-Column-Metadata.csv', 'ACSST1Y2021.S1501-Table-Notes.txt', 'ACSST1Y2019.S1501-Data.csv', 'ACSST1Y2019.S1501-Column-Metadata.csv', 'ACSST1Y2019.S1501-Table-Notes.txt', 'ACSST1Y2018.S1501-Data.csv', 'ACSST1Y2018.S1501-Column-Metadata.csv', 'ACSST1Y2018.S1501-Table-Notes.txt', 'ACSST1Y2017.S1501-Data.csv', 'ACSST1Y2017.S1501-Column-Metadata.csv', 'ACSST1Y2017.S1501-Table-Notes.txt', 'ACSST1Y2016.S1501-Data.csv', 'ACSST1Y2016.S1501-Column-Metadata.csv', 'ACSST1Y2016.S1501-Table-Notes.txt', 'ACSST1Y2015.S1501-Data.csv', 'ACSST1Y2015.S1501-Column-Metadata.csv', 'ACSST1Y2015.S1501-Table-Notes.txt', 'ACSST1Y2014.S1501-Data.csv

In [35]:
cols_attainment = [
    "Geography",
    "pop25plus_total",
    "hs_grad_total",
    "ba_degree_total",
    "grad_degree_total"
]

rename_map = {

    # ---- Educational Attainment (S1501 inconsistencies) ----
    # Total population 25+
    "Total!!Estimate!!Population 25 years and over": "pop25plus_total",
    "Estimate!!Total!!AGE BY EDUCATIONAL ATTAINMENT!!Population 25 years and over": "pop25plus_total",

    # High school graduate
    "Total!!Estimate!!High school graduate (includes equivalency)": "hs_grad_total",
    "Estimate!!Total!!AGE BY EDUCATIONAL ATTAINMENT!!Population 25 years and over!!High school graduate (includes equivalency)": "hs_grad_total",

    # Bachelor's degree
    "Total!!Estimate!!Bachelor's degree": "ba_degree_total",
    "Estimate!!Total!!AGE BY EDUCATIONAL ATTAINMENT!!Population 25 years and over!!Bachelor's degree": "ba_degree_total",

    # Graduate or professional degree
    "Total!!Estimate!!Graduate or professional degree": "grad_degree_total",
    "Estimate!!Total!!AGE BY EDUCATIONAL ATTAINMENT!!Population 25 years and over!!Graduate or professional degree": "grad_degree_total"
}


all_dfs = []

with zipfile.ZipFile(zip_path, "r") as z:
    
    # list all CSVs inside
    csv_files = [
        f for f in z.namelist()
        if f.endswith("-Data.csv")
        and "MACOSX" not in f
        and ".DS_Store" not in f
    ]
    
    for file in csv_files:

        # Extract year and survey type
        # Examples:
        #   ACSST1Y2014.S1401-Data.csv → type=1, year=2014
        #   ACSST5Y2020.S1401-Data.csv → type=5, year=2020
        m = re.search(r'ACSST([15])Y(\d{4})\.S1501-Data\.csv', file)
        if not m:
            continue
        
        survey_type = m.group(1)   # "1" or "5"
        year = int(m.group(2))

        # Apply the ACS rule:
        #   Use 1-year for all years except 2020  
        #   Use 5-year ONLY for 2020  
        if year != 2020 and survey_type == "5":
            continue
        if year == 2020 and survey_type == "1":
            continue
        
        print(f"Loading: {file}  (year={year}, type={survey_type}-year)")

        with z.open(file) as f:
            df = pd.read_csv(f, header=1)

        df = df.iloc[1:].reset_index(drop=True)
        df = df.rename(columns=rename_map)
        
        df = df[[c for c in cols_attainment if c in df.columns]]
        
        df["year"] = year
        all_dfs.append(df)

# Combine everything
df_all_ea = pd.concat(all_dfs, ignore_index=True)
df_all_ea["year"] = df_all_ea["year"].astype(int)
df_all_ea = df_all_ea.sort_values(["year", "Geography"]).reset_index(drop=True)

df_all_ea.head()

Loading: ACSST1Y2024.S1501-Data.csv  (year=2024, type=1-year)
Loading: ACSST1Y2023.S1501-Data.csv  (year=2023, type=1-year)
Loading: ACSST1Y2022.S1501-Data.csv  (year=2022, type=1-year)
Loading: ACSST1Y2021.S1501-Data.csv  (year=2021, type=1-year)
Loading: ACSST1Y2019.S1501-Data.csv  (year=2019, type=1-year)
Loading: ACSST1Y2018.S1501-Data.csv  (year=2018, type=1-year)
Loading: ACSST1Y2017.S1501-Data.csv  (year=2017, type=1-year)
Loading: ACSST1Y2016.S1501-Data.csv  (year=2016, type=1-year)
Loading: ACSST1Y2015.S1501-Data.csv  (year=2015, type=1-year)
Loading: ACSST1Y2014.S1501-Data.csv  (year=2014, type=1-year)
Loading: ACSST1Y2013.S1501-Data.csv  (year=2013, type=1-year)
Loading: ACSST1Y2012.S1501-Data.csv  (year=2012, type=1-year)
Loading: ACSST1Y2011.S1501-Data.csv  (year=2011, type=1-year)
Loading: ACSST1Y2010.S1501-Data.csv  (year=2010, type=1-year)
Loading: ACSST5Y2020.S1501-Data.csv  (year=2020, type=5-year)


/tmp/ipykernel_10884/584672584.py:66: DtypeWarning: Columns (118,119,122,123,124,125,132,133,134,135,136,137,138,139,192,193,194,195,228,229,230,231,234,235,236,237,244,245,374,375,378,379,380,381,388,389,390,391,392,393,394,395,448,449,450,451,484,485,486,487,490,491,492,493,496,497,500,501,630,631,644,645,646,647,648,649,650,651,674,675,676,677,680,681,682,683,704,705,706,707,740,741,742,743,746,747,748,749,750,751,756,757) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, header=1)


,Geography,pop25plus_total,hs_grad_total,ba_degree_total,grad_degree_total,year
0,0500000US01015,78510.0,37.8,9.7,5.0,2010
1,0500000US01043,54033.0,23.8,9.2,5.2,2010
2,0500000US01049,45794.0,41.8,7.9,4.2,2010
3,0500000US01051,53657.0,39.8,13.5,8.1,2010
4,0500000US01055,71575.0,29.0,8.2,4.7,2010


In [36]:
df_all_ea["STCOFIPS"] = df_all_ea["Geography"].str[-5:]

### School Enrollment

https://data.census.gov/table/ACSST5Y2023.S1401?q=S1401&g=010XX00US$0500000

In [37]:
zip_path = "ACSST1Y2010.S1401.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()
    print(files)

['ACSST1Y2024.S1401-Data.csv', 'ACSST1Y2024.S1401-Column-Metadata.csv', 'ACSST1Y2024.S1401-Table-Notes.txt', 'ACSST1Y2023.S1401-Data.csv', 'ACSST1Y2023.S1401-Column-Metadata.csv', 'ACSST1Y2023.S1401-Table-Notes.txt', 'ACSST1Y2022.S1401-Data.csv', 'ACSST1Y2022.S1401-Column-Metadata.csv', 'ACSST1Y2022.S1401-Table-Notes.txt', 'ACSST1Y2021.S1401-Data.csv', 'ACSST1Y2021.S1401-Column-Metadata.csv', 'ACSST1Y2021.S1401-Table-Notes.txt', 'ACSST1Y2019.S1401-Data.csv', 'ACSST1Y2019.S1401-Column-Metadata.csv', 'ACSST1Y2019.S1401-Table-Notes.txt', 'ACSST1Y2018.S1401-Data.csv', 'ACSST1Y2018.S1401-Column-Metadata.csv', 'ACSST1Y2018.S1401-Table-Notes.txt', 'ACSST1Y2017.S1401-Data.csv', 'ACSST1Y2017.S1401-Column-Metadata.csv', 'ACSST1Y2017.S1401-Table-Notes.txt', 'ACSST1Y2016.S1401-Data.csv', 'ACSST1Y2016.S1401-Column-Metadata.csv', 'ACSST1Y2016.S1401-Table-Notes.txt', 'ACSST1Y2015.S1401-Data.csv', 'ACSST1Y2015.S1401-Column-Metadata.csv', 'ACSST1Y2015.S1401-Table-Notes.txt', 'ACSST1Y2014.S1401-Data.csv

In [38]:
cols_enrollment = [
    "Geography",
    "enrolled_total"
]

rename_map = {
    "Estimate!!Total!!Population 3 years and over enrolled in school": "enrolled_total",
    "Total!!Estimate!!Population 3 years and over enrolled in school": "enrolled_total"
}


all_dfs = []

with zipfile.ZipFile(zip_path, "r") as z:
    
    # list all CSVs inside
    csv_files = [
        f for f in z.namelist()
        if f.endswith("-Data.csv")
        and "MACOSX" not in f
        and ".DS_Store" not in f
    ]
    
    for file in csv_files:

        # Extract year and survey type
        # Examples:
        #   ACSST1Y2014.S1401-Data.csv → type=1, year=2014
        #   ACSST5Y2020.S1401-Data.csv → type=5, year=2020
        m = re.search(r'ACSST([15])Y(\d{4})\.S1401-Data\.csv', file)
        if not m:
            continue
        
        survey_type = m.group(1)   # "1" or "5"
        year = int(m.group(2))

        # Apply the ACS rule:
        #   Use 1-year for all years except 2020  
        #   Use 5-year ONLY for 2020  
        if year != 2020 and survey_type == "5":
            continue
        if year == 2020 and survey_type == "1":
            continue
        
        print(f"Loading: {file}  (year={year}, type={survey_type}-year)")

        with z.open(file) as f:
            df = pd.read_csv(f, header=1)

        df = df.iloc[1:].reset_index(drop=True)
        df = df.rename(columns=rename_map)
        
        df = df[[c for c in cols_enrollment if c in df.columns]]
        
        df["year"] = year
        all_dfs.append(df)

# Combine everything
df_all_ee = pd.concat(all_dfs, ignore_index=True)
df_all_ee["year"] = df_all_ee["year"].astype(int)
df_all_ee = df_all_ee.sort_values(["year", "Geography"]).reset_index(drop=True)

df_all_ee.head()



Loading: ACSST1Y2024.S1401-Data.csv  (year=2024, type=1-year)
Loading: ACSST1Y2023.S1401-Data.csv  (year=2023, type=1-year)
Loading: ACSST1Y2022.S1401-Data.csv  (year=2022, type=1-year)
Loading: ACSST1Y2021.S1401-Data.csv  (year=2021, type=1-year)
Loading: ACSST1Y2019.S1401-Data.csv  (year=2019, type=1-year)
Loading: ACSST1Y2018.S1401-Data.csv  (year=2018, type=1-year)
Loading: ACSST1Y2017.S1401-Data.csv  (year=2017, type=1-year)
Loading: ACSST1Y2016.S1401-Data.csv  (year=2016, type=1-year)
Loading: ACSST1Y2015.S1401-Data.csv  (year=2015, type=1-year)
Loading: ACSST1Y2014.S1401-Data.csv  (year=2014, type=1-year)
Loading: ACSST1Y2013.S1401-Data.csv  (year=2013, type=1-year)
Loading: ACSST1Y2012.S1401-Data.csv  (year=2012, type=1-year)
Loading: ACSST1Y2011.S1401-Data.csv  (year=2011, type=1-year)
Loading: ACSST1Y2010.S1401-Data.csv  (year=2010, type=1-year)
Loading: ACSST5Y2020.S1401-Data.csv  (year=2020, type=5-year)


/tmp/ipykernel_10884/111443725.py:48: DtypeWarning: Columns (72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,92,93,128,129,206,207,342,343) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, header=1)


,Geography,enrolled_total,year
0,0500000US01015,30222,2010
1,0500000US01043,20730,2010
2,0500000US01049,16540,2010
3,0500000US01051,18572,2010
4,0500000US01055,23691,2010


In [39]:
df_all_ee["STCOFIPS"] = df_all_ee["Geography"].str[-5:]

### Population Estimates

In [40]:
df1=pd.read_csv('population_estimates/cc-est2019-alldata.csv', encoding="latin1")
df2=pd.read_csv('population_estimates/cc-est2024-alldata.csv', encoding="latin1")

In [41]:
df1 = (df1
 [(df1['AGEGRP'].isin([1,2,3,4,5]))&
  (df1['YEAR']!=1)]
 .groupby(['STATE', 'COUNTY', 'YEAR'])
 .agg({'TOT_POP':'sum'})
 .reset_index()
)

df1['YEAR']=2008+df1['YEAR']

In [42]:
df1["STCOFIPS"] = df1["STATE"].astype(str).str.zfill(2)+df1["COUNTY"].astype(str).str.zfill(3)

In [43]:
df1

,STATE,COUNTY,YEAR,TOT_POP,STCOFIPS
0,1,1,2010,19244,01001
1,1,1,2011,19212,01001
2,1,1,2012,19199,01001
3,1,1,2013,18970,01001
4,1,1,2014,18621,01001
...,...,...,...,...,...
34557,56,45,2016,2110,56045
34558,56,45,2017,2088,56045
34559,56,45,2018,1895,56045
34560,56,45,2019,1859,56045


In [44]:
df2 = (df2
 [(df2['AGEGRP'].isin([1,2,3,4,5]))&
  (~df2['YEAR'].isin([1,2]))]
 .groupby(['STATE', 'COUNTY', 'YEAR'])
 .agg({'TOT_POP':'sum'})
 .reset_index()
)

df2['YEAR']=2018+df2['YEAR']

In [45]:
df2["STCOFIPS"] = df2["STATE"].astype(str).str.zfill(2)+df1["COUNTY"].astype(str).str.zfill(3)

In [46]:
df_pop = pd.concat([df1, df2], ignore_index=True)

In [47]:
df_pop = df_pop.rename(columns={'TOT_POP':'pop_below_25_total',
                                'YEAR':'year'})

In [48]:
df_pop['year']=df_pop['year'].astype(str)
df_all_ee['year']=df_all_ee['year'].astype(str)
df_all_ea['year']=df_all_ea['year'].astype(str)

In [49]:
df_pop.groupby("year").size()

year
2010    3142
2011    3142
2012    3142
2013    3142
2014    3142
2015    3142
2016    3142
2017    3142
2018    3142
2019    3142
2020    3142
2021    3144
2022    3144
2023    3144
2024    3144
dtype: int64

In [50]:
df_all_ee.groupby("year").size()

year
2010     817
2011     821
2012     824
2013     827
2014     827
2015     829
2016     830
2017     836
2018     837
2019     839
2020    3220
2021     840
2022     847
2023     853
2024     860
dtype: int64

In [51]:
df_all_ea.groupby("year").size()

year
2010     817
2011     821
2012     824
2013     827
2014     827
2015     829
2016     830
2017     836
2018     837
2019     839
2020    3220
2021     840
2022     847
2023     853
2024     860
dtype: int64

In [52]:
df_education_combined=(df_pop[['STCOFIPS', 'year', 'pop_below_25_total']]
 .merge(df_all_ee[['STCOFIPS', 'year', 'enrolled_total']], on=['STCOFIPS', 'year'], how='outer')
 .merge(df_all_ea[['STCOFIPS', 'year', 'pop25plus_total', 
                   'hs_grad_total', 'ba_degree_total', 'grad_degree_total']], 
        on=['STCOFIPS', 'year'], how='outer')
)

df_education_combined = df_education_combined[df_education_combined['year']=='2020'].reset_index(drop=True)

In [53]:
df_education_combined=df_education_combined.dropna().reset_index(drop=True)

In [54]:
cols_to_num = [
    'enrolled_total',
    'pop_below_25_total',
    'pop25plus_total',
    'hs_grad_total',
    'ba_degree_total',
    'grad_degree_total'
]

df_education_combined[cols_to_num] = (
    df_education_combined[cols_to_num]
    .apply(pd.to_numeric, errors='coerce')
)

In [55]:
df_education_combined['enrollment_perc'] = \
    df_education_combined['enrolled_total'] / df_education_combined['pop_below_25_total'] * 100

df_education_combined['hs_perc'] = \
    df_education_combined['hs_grad_total'] / df_education_combined['pop25plus_total'] * 100

df_education_combined['ba_perc'] = \
    df_education_combined['ba_degree_total'] / df_education_combined['pop25plus_total'] * 100

df_education_combined['grad_perc'] = \
    df_education_combined['grad_degree_total'] / df_education_combined['pop25plus_total'] * 100

df_education_combined['educational_attainment'] = \
    (df_education_combined['hs_perc'] + 2*df_education_combined['ba_perc'] + 3*df_education_combined['grad_perc']) / 100


In [56]:
df_education_combined = df_education_combined.replace(
    [np.inf, -np.inf],
    0
)

In [57]:
df_education_combined['ea_index'] = (df_education_combined['educational_attainment'] - 0.14) / (2.13 - 0.14) * 10

In [58]:
df_education_combined['ee_index'] = (df_education_combined['enrollment_perc'] - 0) / (147 - 0) * 10

In [59]:
df_education_combined['education_index']= 0.33*df_education_combined['ee_index'] + 0.67*df_education_combined['ea_index']

In [60]:
df_education_combined.sort_values('education_index')

,STCOFIPS,year,pop_below_25_total,enrolled_total,pop25plus_total,hs_grad_total,ba_degree_total,grad_degree_total,enrollment_perc,hs_perc,ba_perc,grad_perc,educational_attainment,ea_index,ee_index,education_index
2671,48301,2020,63.0,0.0,88.0,13.0,0.0,0.0,0.000000,14.772727,0.000000,0.000000,0.147727,0.038831,0.000000,0.026016
2651,48261,2020,145.0,67.0,270.0,39.0,1.0,3.0,46.206897,14.444444,0.370370,1.111111,0.185185,0.227061,3.143326,1.189429
1426,28055,2020,324.0,97.0,1001.0,356.0,9.0,25.0,29.938272,35.564436,0.899101,2.497502,0.448551,1.550510,2.036617,1.710925
2078,39075,2020,17848.0,8777.0,25941.0,8379.0,1824.0,868.0,49.176378,32.300220,7.031340,3.346055,0.564011,2.130707,3.345332,2.531533
454,13141,2020,1945.0,910.0,6842.0,2568.0,470.0,169.0,46.786632,37.532885,6.869336,2.470038,0.586817,2.245310,3.182764,2.554670
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2913,51510,2020,37992.0,31121.0,119957.0,12364.0,37466.0,40573.0,81.914614,10.307027,31.232858,33.822953,1.742416,8.052342,5.572423,7.233969
1955,37135,2020,55922.0,53029.0,91282.0,11449.0,23792.0,31753.0,94.826723,12.542451,26.064284,34.785609,1.690278,7.790344,6.450797,7.348294
1808,35028,2020,5628.0,4754.0,13466.0,1341.0,3394.0,5704.0,84.470505,9.958414,25.204218,42.358533,1.874424,8.715701,5.746293,7.735796
2824,51013,2020,62594.0,49956.0,174580.0,13788.0,61669.0,70725.0,79.809566,7.897812,35.324207,40.511513,2.000808,9.350792,5.429222,8.056674


In [61]:
df_education_combined.to_parquet('education_combined.parquet')

# Health

### 2000-2019

https://ghdx.healthdata.org/record/ihme-data/united-states-life-expectancy-by-county-race-ethnicity-2000-2019

In [62]:
outer_zip = "health_2000_2019.zip"
inner_zip_path = "health_2000_2019/IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2015_2019.zip"

with zipfile.ZipFile(outer_zip, "r") as outer:
    # Open inner ZIP as bytes
    with outer.open(inner_zip_path) as inner_bytes:
        # Now open the inner ZIP from those bytes
        with zipfile.ZipFile(inner_bytes, "r") as inner:
            print(inner.namelist())

['IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2015_BOTH_Y2022M06D16.CSV', 'IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2015_FEMALE_Y2022M06D16.CSV', 'IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2015_MALE_Y2022M06D16.CSV', 'IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2016_BOTH_Y2022M06D16.CSV', 'IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2016_FEMALE_Y2022M06D16.CSV', 'IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2016_MALE_Y2022M06D16.CSV', 'IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2017_BOTH_Y2022M06D16.CSV', 'IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2017_FEMALE_Y2022M06D16.CSV', 'IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2017_MALE_Y2022M06D16.CSV', 'IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2018_BOTH_Y2022M06D16.CSV', 'IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2018_FEMALE_Y2022M06D16.CSV', 'IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2018_MALE_Y2022M06D16.CSV', 'IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2019_BOTH_Y2022M06D16.CSV', 'IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_MX_2019_FEMALE_Y2022M06D16.C

In [63]:
import zipfile
import pandas as pd

outer_zip = "health_2000_2019.zip"

# All LT windows - life expectancy (there is also death)
lt_inner_zips = [
    "health_2000_2019/IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2000_2004.zip",
    "health_2000_2019/IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2005_2009.zip",
    "health_2000_2019/IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2010_2014.zip",
    "health_2000_2019/IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2015_2019.zip"
]

dfs = []

with zipfile.ZipFile(outer_zip, "r") as outer:
    for inner_zip_path in lt_inner_zips:
        print("\nProcessing inner ZIP:", inner_zip_path)

        with outer.open(inner_zip_path) as inner_bytes:
            with zipfile.ZipFile(inner_bytes, "r") as inner:

                for filename in inner.namelist():
                    # Only load BOTH-sex files
                    if ("BOTH" in filename) and filename.lower().endswith(".csv"):
                        print("   Loading:", filename)

                        with inner.open(filename) as f:
                            df = pd.read_csv(f)
                            dfs.append(df)

# Combine all windows
df_all_lt = pd.concat(dfs, ignore_index=True)

print("\n=== FINAL RESULT ===")
df_all_lt.head()
print("Rows:", len(df_all_lt))
print("Columns:", df_all_lt.columns.tolist())




Processing inner ZIP: health_2000_2019/IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2000_2004.zip
   Loading: IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2000_BOTH_Y2022M06D16.CSV
   Loading: IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2001_BOTH_Y2022M06D16.CSV
   Loading: IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2002_BOTH_Y2022M06D16.CSV
   Loading: IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2003_BOTH_Y2022M06D16.CSV
   Loading: IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2004_BOTH_Y2022M06D16.CSV

Processing inner ZIP: health_2000_2019/IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2005_2009.zip
   Loading: IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2005_BOTH_Y2022M06D16.CSV
   Loading: IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2006_BOTH_Y2022M06D16.CSV
   Loading: IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2007_BOTH_Y2022M06D16.CSV
   Loading: IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2008_BOTH_Y2022M06D16.CSV
   Loading: IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2009_BOTH_Y2022M06D16.CSV

Process

In [64]:
# convert fips to string
df_all_lt["fips"] = df_all_lt["fips"].astype(pd.Int64Dtype())
df_all_lt["fips"] = df_all_lt["fips"].astype("string")

# replace national value
df_all_lt.loc[df_all_lt["location_name"] == "United States of America", "fips"] = "00000"

# zero-pad to 5 characters
df_all_lt["fips"] = df_all_lt["fips"].apply(lambda x: x.zfill(5) if isinstance(x, str) and x.isnumeric() else x)


In [65]:
df_all_lt=(df_all_lt[(df_all_lt['age_name']=='<1 year')&
                     (df_all_lt['race_name']=='Total')]
           .rename(columns={'fips':'STCOFIPS',
                            'measure_name':'Description',
                            'metric_name':'Unit',
                            'val':'value'})
           [['STCOFIPS', 'Description', 'Unit', 'year', 'value', 'upper', 'lower']]
           .reset_index(drop=True)
          )

In [66]:
df_all_lt

,STCOFIPS,Description,Unit,year,value,upper,lower
0,00000,Life expectancy,Number,2000,76.780325,76.806569,76.751931
1,00001,Life expectancy,Number,2000,74.364763,74.569500,74.154499
2,01001,Life expectancy,Number,2000,74.722384,75.317537,74.148346
3,01003,Life expectancy,Number,2000,76.726039,77.152319,76.340126
4,01005,Life expectancy,Number,2000,73.993461,74.626864,73.354724
...,...,...,...,...,...,...,...
64195,56037,Life expectancy,Number,2019,78.384155,79.168522,77.691659
64196,56039,Life expectancy,Number,2019,85.905597,86.943858,85.025816
64197,56041,Life expectancy,Number,2019,77.886506,78.814331,76.933296
64198,56043,Life expectancy,Number,2019,78.532334,79.581140,77.524879


### 2020-2025

https://www.countyhealthrankings.org/health-data

In [67]:
all_files = [
    "health_2020_2025/2020 County Health Rankings Data - v2.xlsx",
    "health_2020_2025/2021 County Health Rankings Data - v1.xlsx",
    "health_2020_2025/2022 County Health Rankings Data - v1.xlsx",
    "health_2020_2025/2023 County Health Rankings Data - v2.xlsx",
    "health_2020_2025/2024_county_health_release_data_-_v1.xlsx",
    "health_2020_2025/2025 County Health Rankings Data - v3.xlsx"
]

dfs = []

for file_path in all_files:
    print("Reading:", file_path)

    # Extract YEAR from file name
    year = int(re.findall(r"20\d\d", file_path)[2])

    # Select sheet based on year
    sheet = 5 if year in [2020, 2021, 2022, 2023] else 3

    df = pd.read_excel(file_path, sheet_name=sheet, header=1)

    cols = ['FIPS', 'State', 'County', 'Life Expectancy', '95% CI - Low', '95% CI - High']
    df = df[cols]
    df['year']=year

    dfs.append(df)

# Combine all years
df_all_years = pd.concat(dfs, ignore_index=True)

print(df_all_years.head())
print("Rows:", len(df_all_years))

Reading: health_2020_2025/2020 County Health Rankings Data - v2.xlsx
Reading: health_2020_2025/2021 County Health Rankings Data - v1.xlsx
Reading: health_2020_2025/2022 County Health Rankings Data - v1.xlsx
Reading: health_2020_2025/2023 County Health Rankings Data - v2.xlsx
Reading: health_2020_2025/2024_county_health_release_data_-_v1.xlsx
Reading: health_2020_2025/2025 County Health Rankings Data - v3.xlsx
   FIPS    State   County  Life Expectancy  95% CI - Low  95% CI - High  year
0  1000  Alabama      NaN        75.416194     75.332867      75.499522  2020
1  1001  Alabama  Autauga        76.879477     76.112671      77.646283  2020
2  1003  Alabama  Baldwin        78.450258     78.058764      78.841753  2020
3  1005  Alabama  Barbour        75.341935     74.158832      76.525038  2020
4  1007  Alabama     Bibb        73.571820     72.290275      74.853366  2020
Rows: 19183


In [68]:
df_all_years["FIPS"] = df_all_years["FIPS"].astype(pd.Int64Dtype())
df_all_years["FIPS"] = df_all_years["FIPS"].astype("string")

# zero-pad to 5 characters
df_all_years["FIPS"] = df_all_years["FIPS"].apply(lambda x: x.zfill(5) if isinstance(x, str) and x.isnumeric() else x)

In [69]:
df_all_years = (
    df_all_years
        .rename(columns={
            'FIPS':'STCOFIPS',
            'Life Expectancy':'value',
            '95% CI - Low':'lower',
            '95% CI - High':'upper'
        })
        .assign(
            Description="Life expectancy",
            Unit="Number"      # or whatever is correct
        )
        [['STCOFIPS', 'Description', 'Unit', 'year', 'value', 'upper', 'lower']]
        .reset_index(drop=True)
)


In [70]:
df_all_lt

,STCOFIPS,Description,Unit,year,value,upper,lower
0,00000,Life expectancy,Number,2000,76.780325,76.806569,76.751931
1,00001,Life expectancy,Number,2000,74.364763,74.569500,74.154499
2,01001,Life expectancy,Number,2000,74.722384,75.317537,74.148346
3,01003,Life expectancy,Number,2000,76.726039,77.152319,76.340126
4,01005,Life expectancy,Number,2000,73.993461,74.626864,73.354724
...,...,...,...,...,...,...,...
64195,56037,Life expectancy,Number,2019,78.384155,79.168522,77.691659
64196,56039,Life expectancy,Number,2019,85.905597,86.943858,85.025816
64197,56041,Life expectancy,Number,2019,77.886506,78.814331,76.933296
64198,56043,Life expectancy,Number,2019,78.532334,79.581140,77.524879


In [71]:
df_all_years

,STCOFIPS,Description,Unit,year,value,upper,lower
0,01000,Life expectancy,Number,2020,75.416194,75.499522,75.332867
1,01001,Life expectancy,Number,2020,76.879477,77.646283,76.112671
2,01003,Life expectancy,Number,2020,78.450258,78.841753,78.058764
3,01005,Life expectancy,Number,2020,75.341935,76.525038,74.158832
4,01007,Life expectancy,Number,2020,73.571820,74.853366,72.290275
...,...,...,...,...,...,...,...
19178,56037,Life expectancy,Number,2025,74.630018,75.492918,73.767118
19179,56039,Life expectancy,Number,2025,87.576928,89.611733,85.542123
19180,56041,Life expectancy,Number,2025,74.946119,76.407310,73.484927
19181,56043,Life expectancy,Number,2025,76.072057,78.328182,73.815933


### Combine

In [72]:
df_combined_health=pd.concat([df_all_lt, df_all_years], ignore_index=True)
df_combined_health=df_combined_health.sort_values(['STCOFIPS', 'year']).reset_index(drop=True)

In [73]:
df_combined_health.to_parquet('health_combined.parquet')

In [74]:
df_combined_health.value.max()

113.4621372

In [75]:
df_combined_health[df_combined_health['value']>100]

,STCOFIPS,Description,Unit,year,value,upper,lower
2835,02013,Life expectancy,Number,2021,113.462137,159.000095,67.924179
2836,02013,Life expectancy,Number,2022,112.503856,156.850296,68.157415
2837,02013,Life expectancy,Number,2023,112.503856,156.850296,68.157415
6769,06051,Life expectancy,Number,2021,101.598246,112.119445,91.077047
6770,06051,Life expectancy,Number,2022,103.309976,114.387814,92.232138
6771,06051,Life expectancy,Number,2023,103.309976,114.387814,92.232138
9114,08113,Life expectancy,Number,2020,104.740145,126.150279,83.330011
9115,08113,Life expectancy,Number,2021,102.963081,123.150026,82.776135


In [76]:
lambda_penalty = 0.3
df_combined_health["value_adj"] = df_combined_health["value"] - lambda_penalty * (df_combined_health["upper"] - df_combined_health["lower"])


In [77]:
# raw index
df_combined_health["health_index"] = (
    df_combined_health.groupby("year")["value"]
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()) * 10)
)

# adjusted index using CI midpoint
df_combined_health["health_index_adj"] = (
    df_combined_health.groupby("year")["value_adj"]
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()) * 10)
)

In [78]:
df_combined_health

,STCOFIPS,Description,Unit,year,value,upper,lower,value_adj,health_index,health_index_adj
0,00000,Life expectancy,Number,2000,76.780325,76.806569,76.751931,76.763934,5.489093,5.820089
1,00000,Life expectancy,Number,2001,76.875631,76.902606,76.847716,76.859164,5.481075,5.806743
2,00000,Life expectancy,Number,2002,76.940289,76.968955,76.913377,76.923616,5.491447,5.795039
3,00000,Life expectancy,Number,2003,77.116172,77.142342,77.090351,77.100575,5.463186,5.753485
4,00000,Life expectancy,Number,2004,77.493153,77.520774,77.463488,77.475967,5.371054,5.655058
...,...,...,...,...,...,...,...,...,...,...
83378,56045,Life expectancy,Number,2021,81.965286,83.916385,80.014187,80.794627,3.578525,5.659226
83379,56045,Life expectancy,Number,2022,80.669458,82.647703,78.691214,79.482512,3.812320,5.493150
83380,56045,Life expectancy,Number,2023,80.669458,82.647703,78.691214,79.482512,3.812320,5.493150
83381,56045,Life expectancy,Number,2024,79.152208,80.986685,77.317730,78.051521,5.405095,6.110210


In [79]:
df_combined_health.groupby('year').size()

year
2000    3210
2001    3210
2002    3210
2003    3210
2004    3210
2005    3210
2006    3210
2007    3210
2008    3210
2009    3210
2010    3210
2011    3210
2012    3210
2013    3210
2014    3210
2015    3210
2016    3210
2017    3210
2018    3210
2019    3210
2020    3193
2021    3193
2022    3193
2023    3193
2024    3201
2025    3210
dtype: int64

In [80]:
df_combined_health.to_parquet('health_combined.parquet')